# Document Question Answering System using Retrieval-Augmented Generation (RAG)

This project builds a simple RAG system that answers questions from a custom PDF document using Sentence Transformers, FAISS, and FLAN-T5.

## 1. Import Libraries

First, we import all the libraries required for the project. These libraries help in reading PDF files, splitting text into chunks, creating embeddings, storing vectors in FAISS, and generating answers using a language model.

In [1]:
import PyPDF2

from sentence_transformers import SentenceTransformer

from langchain_text_splitters import RecursiveCharacterTextSplitter

import faiss
import numpy as np

from transformers import pipeline

## 2. Load PDF

In this step, the PDF document is loaded and its text is extracted. This extracted text will be used for further processing in the RAG pipeline.

In [2]:
pdf_path = "data/resume.pdf"

text = ""

with open(pdf_path, "rb") as file:
    reader = PyPDF2.PdfReader(file)

    for page in reader.pages:
        text += page.extract_text()

## 3. Text Chunking

The extracted text is divided into smaller chunks. Chunking makes it easier to search for relevant information and improves the accuracy of retrieval.

In [3]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_text(text)

In [4]:
print(f"Total Chunks: {len(chunks)}")

Total Chunks: 10


## 4. Generate Embeddings

Each text chunk is converted into a numerical vector using a sentence transformer model. These embeddings capture the meaning of the text and are used for similarity search.

In [5]:
#Load embedding model

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: 34d43ab0-6005-41cd-9def-41cdd8c7e1ea)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: f5b6247b-553e-4078-85c6-daff46df55d3)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-M

In [6]:
#Generate vectors

embeddings = embedding_model.encode(
    chunks,
    convert_to_numpy=True
).astype("float32")

In [7]:
print(f"Embedding Shape: {embeddings.shape}")

Embedding Shape: (10, 384)


## 5. Store in FAISS

The generated embeddings are stored in a FAISS index. This allows the system to quickly find the most relevant text chunks for a user's question.

In [8]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

In [9]:
print(f"Vectors Stored: {index.ntotal}")

Vectors Stored: 10


## 6. Load LLM

The language model is loaded to generate answers. It uses the retrieved context from the document to produce responses based on the user's question.

In [10]:
generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-base"
)

Device set to use cpu


## 7. Interactive Function

This function combines all the RAG steps. It takes a user question, retrieves the most relevant chunks from the document, and generates an answer using the language model.

In [11]:
def ask_question(question):

    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True
    ).astype("float32")

    distances, indices = index.search(query_embedding, 3)

    context = ""

    for idx in indices[0]:
        context += chunks[idx] + "\n"

    print("Retrieved Context:")
    print(context)

    prompt = f"""
You are a helpful assistant.

Answer ONLY using the information provided in the context.

If the answer is not available in the context, reply:
"I could not find this information in the document."

Context:
{context}

Question:
{question}

Answer:
"""

    answer = generator(
        prompt,
        max_new_tokens=100
    )

    return answer[0]["generated_text"]

## 8. Test the System

Finally, different questions are asked to check whether the RAG system retrieves the correct information and generates accurate answers from the document.

In [12]:
ask_question("What programming languages are mentioned?")

Retrieved Context:
Databases: MySQL, MongoDB  
Developer Tools: VS Code, GitHub, Jupyter Notebook, Postman , Kiro  
Relevant Coursework: Data Structures & Algorithms, OOPs, Operating Systems, AWS  
PROJECTS  
 
1. GatiSetu – AI-Powered Smart Traffic Management  
 
  
 
 (Smart India Hackathon 2025 & AMD Slingshot Hackathon)  
 Tech Stack: Python, Computer Vision, YOLO, OpenCV, FastAPI  
• Developed an AI -powered smart traffic management system to optimize urban traffic flow using real -time video 
analytics.
Tech Stack: AWS, Amazon Lex V2, Lambda, DynamoDB, API Gateway, Step Functions  
• Built a voice -first AI assistant for Elderline (14567) to help senior citizens access government services.  
• Designed multilingual conversational workflows using AWS serverless services.  
• Enabled accessibility for users without smartphones or internet connectivity.  
3. Customer Intelligence System  
(Celebal Technologies Internship)  
Tech Stack: Python, Pandas, Scikit -learn, XGBoost, LightGB

'Java, Python, JavaScript, C, SQL, R, HTML, CSS Frameworks & Libraries: TensorFlow, Scikit -learn, Pandas, NumPy, FastAPI, Flask, Streamlit Databases: MySQL, MongoDB'

In [13]:
ask_question("What skills are mentioned?")

Retrieved Context:
Elderline (14567).  
• Gen AI Academy 2.0  | Completed hands -on training in Machine Learning, Generative AI, Apache Spark, 
and Google Cloud AI APIs.   
• Java Programming – Intellipaat  | Completed training in Java, OOP, Data Structures, and Algorithms.  
 
DOMAIN / TECHNICAL SKILLS  
Languages: Java, Python, JavaScript, C, SQL, R, HTML, CSS  
Frameworks & Libraries: TensorFlow, Scikit -learn, Pandas, NumPy, FastAPI, Flask, Streamlit  
Databases: MySQL, MongoDB
LEADERSHIP/EXTRACURRICULAR  
 
• Participated in Smart India Hackathon (SIH) 2025, AMD Slingshot Hackathon, and AI For Bharat Hackathon, collaborating 
on AI -driven solutions.  
• Continuously practice Data Structures & Algorithms and build AI/ML projects to strengthen problem -solving skills.  
• Actively learning Generative AI, Machine Learning, and Cloud technologies through hands -on projects and technical 
programs.
Secondary (Class X), MP Board                                                          

'Java, Python, JavaScript, C, SQL, R, HTML, CSS Frameworks & Libraries: TensorFlow, Scikit -learn, Pandas, NumPy, FastAPI, Flask, Streamlit Databases: MySQL, MongoDB'

In [14]:
ask_question("Who is the author?")

Retrieved Context:
LEADERSHIP/EXTRACURRICULAR  
 
• Participated in Smart India Hackathon (SIH) 2025, AMD Slingshot Hackathon, and AI For Bharat Hackathon, collaborating 
on AI -driven solutions.  
• Continuously practice Data Structures & Algorithms and build AI/ML projects to strengthen problem -solving skills.  
• Actively learning Generative AI, Machine Learning, and Cloud technologies through hands -on projects and technical 
programs.
Agentic AI workflows as part of hands -on industry projects.  
 
 ACHIE VEMENTS  / CERTIFICATIONS  
• Smart India Hackathon (SIH) 2025  – National Level Finalist | Developed GatiSetu , an AI -powered 
smart traffic management solution.  
• AMD Slingshot Hackathon  – National Level Shortlisted | Built GatiSetuAI , an edge AI traffic 
intelligence platform for congestion prediction.  
• AI For Bharat Hackathon  – National Level Shortlisted | Developed Sahara AI , a voice -first AI assistant for
Siddharth Gupta  
 
hey.siddharthgupta @gmail.com ⋄ (+91)

'Siddharth Gupta'

In [15]:
ask_question("What projects are described?")

Retrieved Context:
Databases: MySQL, MongoDB  
Developer Tools: VS Code, GitHub, Jupyter Notebook, Postman , Kiro  
Relevant Coursework: Data Structures & Algorithms, OOPs, Operating Systems, AWS  
PROJECTS  
 
1. GatiSetu – AI-Powered Smart Traffic Management  
 
  
 
 (Smart India Hackathon 2025 & AMD Slingshot Hackathon)  
 Tech Stack: Python, Computer Vision, YOLO, OpenCV, FastAPI  
• Developed an AI -powered smart traffic management system to optimize urban traffic flow using real -time video 
analytics.
LEADERSHIP/EXTRACURRICULAR  
 
• Participated in Smart India Hackathon (SIH) 2025, AMD Slingshot Hackathon, and AI For Bharat Hackathon, collaborating 
on AI -driven solutions.  
• Continuously practice Data Structures & Algorithms and build AI/ML projects to strengthen problem -solving skills.  
• Actively learning Generative AI, Machine Learning, and Cloud technologies through hands -on projects and technical 
programs.
Secondary (Class X), MP Board                              

'GatiSetu – AI-Powered Smart Traffic Management (Smart India Hackathon 2025 & AMD Slingshot Hackathon)'

In [16]:
ask_question("Summarize the document.")

Retrieved Context:
Tech Stack: Python, Pandas, Scikit -learn, XGBoost, LightGBM, Streamlit  
• Built an end -to-end machine learning pipeline for customer analytics using real -world data.  
• Performed data preprocessing, feature engineering, EDA, model training, and evaluation.  
• Developed classification and clustering models to generate actionable business insights.  
LEADERSHIP/EXTRACURRICULAR
Tech Stack: AWS, Amazon Lex V2, Lambda, DynamoDB, API Gateway, Step Functions  
• Built a voice -first AI assistant for Elderline (14567) to help senior citizens access government services.  
• Designed multilingual conversational workflows using AWS serverless services.  
• Enabled accessibility for users without smartphones or internet connectivity.  
3. Customer Intelligence System  
(Celebal Technologies Internship)  
Tech Stack: Python, Pandas, Scikit -learn, XGBoost, LightGBM, Streamlit
feature engineering, model training, and evaluation on real -world datasets.  
• Developed and comp

'I could not find this information in the document.'

In [17]:
ask_question("Should we hire him?")

Retrieved Context:
feature engineering, model training, and evaluation on real -world datasets.  
• Developed and compared regression, classification, clustering, and deep learning models using Python, Scikit -
learn, XGBoost, LightGBM, TensorFlow/PyTorch, and evaluated their performance using appropriate metrics . 
• Implemented Generative AI applications including Retrieval -Augmented Generation (RAG), LangChain, LangGraph, and 
Agentic AI workflows as part of hands -on industry projects.
Agentic AI workflows as part of hands -on industry projects.  
 
 ACHIE VEMENTS  / CERTIFICATIONS  
• Smart India Hackathon (SIH) 2025  – National Level Finalist | Developed GatiSetu , an AI -powered 
smart traffic management solution.  
• AMD Slingshot Hackathon  – National Level Shortlisted | Built GatiSetuAI , an edge AI traffic 
intelligence platform for congestion prediction.  
• AI For Bharat Hackathon  – National Level Shortlisted | Developed Sahara AI , a voice -first AI assistant for
Second

'I could not find this information in the document.'

In [18]:
ask_question("Is it a good candidate?")

Retrieved Context:
Agentic AI workflows as part of hands -on industry projects.  
 
 ACHIE VEMENTS  / CERTIFICATIONS  
• Smart India Hackathon (SIH) 2025  – National Level Finalist | Developed GatiSetu , an AI -powered 
smart traffic management solution.  
• AMD Slingshot Hackathon  – National Level Shortlisted | Built GatiSetuAI , an edge AI traffic 
intelligence platform for congestion prediction.  
• AI For Bharat Hackathon  – National Level Shortlisted | Developed Sahara AI , a voice -first AI assistant for
LEADERSHIP/EXTRACURRICULAR  
 
• Participated in Smart India Hackathon (SIH) 2025, AMD Slingshot Hackathon, and AI For Bharat Hackathon, collaborating 
on AI -driven solutions.  
• Continuously practice Data Structures & Algorithms and build AI/ML projects to strengthen problem -solving skills.  
• Actively learning Generative AI, Machine Learning, and Cloud technologies through hands -on projects and technical 
programs.
Secondary (Class X), MP Board                            

'I could not find this information in the document.'

In [19]:
ask_question("What are his achievements?")

Retrieved Context:
Databases: MySQL, MongoDB  
Developer Tools: VS Code, GitHub, Jupyter Notebook, Postman , Kiro  
Relevant Coursework: Data Structures & Algorithms, OOPs, Operating Systems, AWS  
PROJECTS  
 
1. GatiSetu – AI-Powered Smart Traffic Management  
 
  
 
 (Smart India Hackathon 2025 & AMD Slingshot Hackathon)  
 Tech Stack: Python, Computer Vision, YOLO, OpenCV, FastAPI  
• Developed an AI -powered smart traffic management system to optimize urban traffic flow using real -time video 
analytics.
LEADERSHIP/EXTRACURRICULAR  
 
• Participated in Smart India Hackathon (SIH) 2025, AMD Slingshot Hackathon, and AI For Bharat Hackathon, collaborating 
on AI -driven solutions.  
• Continuously practice Data Structures & Algorithms and build AI/ML projects to strengthen problem -solving skills.  
• Actively learning Generative AI, Machine Learning, and Cloud technologies through hands -on projects and technical 
programs.
Siddharth Gupta  
 
hey.siddharthgupta @gmail.com ⋄ (+91) 6

'Developed an AI -powered smart traffic management system to optimize urban traffic flow using real - time video analytics. • Continuously practice Data Structures & Algorithms and build AI/ML projects to strengthen problem -solving skills. • Actively learning Generative AI, Machine Learning, and Cloud technologies through hands -on projects and technical programs.'

## Conclusion

In this project, I built a simple Retrieval-Augmented Generation (RAG) system that can answer questions from a custom PDF document. The document was processed by extracting text, splitting it into smaller chunks, generating embeddings, and storing them in a FAISS vector database. When a question is asked, the system retrieves the most relevant chunks and uses a language model to generate an answer based on that context.

This project helped me understand how RAG combines information retrieval with language models to provide more accurate and context-aware responses. It also gave me practical experience with embeddings, vector databases, and document-based question answering, which are important concepts in modern Generative AI applications.
